# Week 10｜課前設定 (setup)

1. 在 VS Code 的終端機（已啟動 `.venv`）執行：

    `pip install jieba rank_bm25 scikit-learn`
    
    `pip install sentence-transformers`

2. 下載模型 從自主學習雲端學院下載模型 從數位學習平台下載 `bge-small-zh-v1.5.zip`，解壓縮到專案的 models/ 資料夾。 (檔案較大，優先執行。)

3. 按上方的 **Run All**。 第 2 節開始前回來確認最後一格已經印出「全部完成」。


這份 notebook 會做 5 件事：設定路徑、設定 `.gitignore`、更新 `requirements.txt`、下載句向量模型、預先計算 chunks 的向量。只需要執行一次；重複執行也不會造成問題。

如果出現 `dependency conflicts` 的紅字，但沒有顯示安裝失敗，可以忽略。

## 步驟 1｜設定專案路徑


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import os

# 檢查目前工作目錄
current_dir = os.getcwd()
print(f"目前工作目錄: {current_dir}")

# 取得目前 Notebook 的絕對路徑
notebook_dir = os.path.abspath('')

# 往上推兩層取得根目錄的絕對路徑
# project_root = os.path.dirname(os.path.dirname(notebook_dir))

# 往上推一層取得根目錄的絕對路徑
project_root = os.path.dirname(notebook_dir)

# 組合出 data/processed 的絕對路徑
PROCESSED_DATA_DIR = Path(os.path.join(project_root, "data", "processed"))

ROOT = Path(project_root)
print(f"專案根目錄: {ROOT}")
assert PROCESSED_DATA_DIR.exists(), "找不到 data/processed。請檢查上面 project_root 是往上推兩層還是一層，並確認第 8 週的 chunks 已經產生。"
print(" 找到 data/processed")

目前工作目錄: c:\Users\user\Desktop\weiwei\U12627006-mcut\notebooks
專案根目錄: c:\Users\user\Desktop\weiwei\U12627006-mcut
 找到 data/processed


## 步驟 2｜設定 `.gitignore` 


今天會產生兩種**不應該上傳到 GitHub** 的檔案：

| 資料夾 | 內容 | 為什麼不上傳 |
|---|---|---|
| `models/` | 句向量模型（離線版本，約 100 MB） | 接近 GitHub 單一檔案 100 MB 的上限，push 可能被拒絕 |
| `data/embeddings/` | chunks 的向量檔 | 可以隨時重新計算，不需要版本控制 |

下一格會檢查專案根目錄的 `.gitignore`，如果還沒有這兩行，就自動加上。你也可以自己打開 `.gitignore`，手動加入：

```
models/
data/embeddings/
```

In [2]:
gitignore = ROOT / ".gitignore"
wanted = ["models/", "data/embeddings/"]

existing = gitignore.read_text(encoding="utf-8").splitlines() if gitignore.exists() else []
missing = [w for w in wanted if w not in [line.strip() for line in existing]]

if missing:
    with gitignore.open("a", encoding="utf-8") as f:
        f.write("\n# Week 10：模型與向量檔不上傳\n" + "\n".join(missing) + "\n")
    print("已加入 .gitignore：", missing)
else:
    print(" .gitignore 已經包含 models/ 和 data/embeddings/")

已加入 .gitignore： ['models/', 'data/embeddings/']


## 步驟 3｜設定  `requirements.txt`

`requirements.txt` 記錄這個專案需要哪些套件。別人（或你自己換一台電腦）clone 專案之後，執行 `pip install -r requirements.txt` 就能安裝相同的套件。今天新增了套件，所以要更新這個檔案。

下一格會把**本週 `pip install` 的套件**加到專案根目錄 `requirements.txt` 的最後；已經列出的不會重複加入。你也可以自己打開檔案，手動加入：

```
jieba
rank_bm25
scikit-learn
sentence-transformers
```

In [3]:
req_path = ROOT / "requirements.txt"
week10_packages = ["jieba", "rank_bm25", "scikit-learn", "sentence-transformers"]

try:
    lines = req_path.read_text(encoding="utf-8").splitlines()
    # 取出每一行的套件名稱（去掉 ==、>= 等版本號），用來檢查是否已經列出
    listed = [line.replace(">", "=").replace("<", "=").split("=")[0].strip().lower() for line in lines]
    missing = [p for p in week10_packages if p.lower() not in listed]

    if missing:
        with req_path.open("a", encoding="utf-8") as f:
            f.write("\n" + "\n".join(missing) + "\n")
        print("已加入 requirements.txt：", missing)
    else:
        print("✅ requirements.txt 已經包含本週安裝的套件")
except Exception as e:
    print("無法自動更新 requirements.txt，請自己打開檔案，手動加入上面五行。原因：", e)

已加入 requirements.txt： ['jieba', 'rank_bm25', 'scikit-learn', 'sentence-transformers']


## 步驟 4｜下載並載入句向量模型

- 如果專案根目錄中有 `models/bge-small-zh-v1.5/`（老師提供的離線版本），會直接使用，不需要下載。
- 否則會從 Hugging Face 下載（約 100 MB）。下載一次之後會存在電腦裡，以後載入只要幾秒鐘。
- Windows 上可能出現關於 symlinks 的黃色警告，可以忽略。
- 如果下載失敗，先和同組同學共用一台電腦，並告訴老師。

In [4]:
import time
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-zh-v1.5"
LOCAL_MODEL = ROOT / "models" / "bge-small-zh-v1.5"

t0 = time.time()
model = SentenceTransformer(str(LOCAL_MODEL) if LOCAL_MODEL.exists() else MODEL_NAME)
print("模型來源：", "本機資料夾 models/" if LOCAL_MODEL.exists() else "Hugging Face（或本機快取）")
print(f"✅ 步驟 5 完成，載入時間：{time.time() - t0:.1f} 秒")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

模型來源： 本機資料夾 models/
✅ 步驟 5 完成，載入時間：0.5 秒


## 步驟 5｜預先計算 chunks 的向量

把第 8 週的所有 chunks 變成句向量，存到 `data/embeddings/week10_chunk_emb.npy`。第 2 節會說明這一步在做什麼；先算好，到時候就不必等待。

In [5]:
CLUSTERS = ["Academic_Affairs", "Library_Information", "Student_Affairs"]

rows = []
for cluster in CLUSTERS:
    for path in sorted((PROCESSED_DATA_DIR / cluster).glob("cleaned_*_processed.json")):
        chunks = json.loads(path.read_text(encoding="utf-8"))
        doc = path.stem.removeprefix("cleaned_").removesuffix("_processed")
        for i, c in enumerate(chunks):
            meta = c.get("metadata", {})
            rows.append({
                "chunk_id": f"{doc}#{i}",
                "cluster": meta.get("cluster", cluster),
                "source_file": meta.get("source_file", path.name),
                "heading": " / ".join(meta.get("headings") or []),
                "text": c["text"],
            })

df = pd.DataFrame(rows)
assert len(df) > 0, "沒有讀到任何 chunk，請檢查 data/processed/<類別>/cleaned_*_processed.json 是否存在。"
texts = df["text"].tolist()

print(f"共載入 {len(df)} 個 chunks，來自 {df['source_file'].nunique()} 份文件")

共載入 345 個 chunks，來自 9 份文件


In [6]:
EMB_PATH = ROOT / "data" / "embeddings" / "week10_chunk_emb.npy"

if EMB_PATH.exists() and np.load(EMB_PATH).shape[0] == len(texts):
    print("向量檔已經存在：", EMB_PATH)
else:
    t0 = time.time()
    chunk_emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=16)
    EMB_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(EMB_PATH, chunk_emb)
    print(f"計算時間：{time.time() - t0:.1f} 秒，已儲存：{EMB_PATH}")

print("全部完成。可以關閉這份 notebook（關閉前可按 Restart 或關掉分頁，釋放記憶體）。")

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

計算時間：15.4 秒，已儲存：c:\Users\user\Desktop\weiwei\U12627006-mcut\data\embeddings\week10_chunk_emb.npy
全部完成。可以關閉這份 notebook（關閉前可按 Restart 或關掉分頁，釋放記憶體）。
